# Введение. Полносвязные слои. Функции активации (ноутбук)

> Начнем осваивать библиотеку `PyTorch`.

## План ноутбука

1. Установка `PyTorch`
1. Введение в `PyTorch`
1. Полносвязные слои и функции активации в `PyTorch`
1. Градиентный спуск своими руками
1. Подробнее про внутреннее устройство тензора

## Установка `PyTorch`

Мы будем использовать библиотеку для глубинного обучения `PyTorch`, ее можно не устанавливать, можно пользоваться сайтами [Kaggle](kaggle.com) и [Google Colab](colab.research.google.com/) для обучения в облаке (или с учителем?).

Чтобы установить `PyTorch` локально себе на компьютер нужно ответить на два вопроса - какая у вас операционная система и есть ли у вас дискретная видеокарта (GPU) и если есть, то какого производителя. В зависимости от ваших ответов мы получаем три варианта по операционной системе - Linux, Mac и Windows; три варианта по дискретной видеокарте - нет видеокарты (доступен только центральный процессор CPU), есть видеокарта от Nvidia или есть видеокарта от AMD (это производитель именно чипа, конечный вендор может быть другой, например, ASUS, MSI, Palit). Работа с PyTorch с видеокартой от AMD это экзотика, которая выходит за рамки нашего курса, поэтому рассмотрим только варианты *нет видеокарты*/*есть видеокарта от Nvidia*.


Выберите на [сайте](https://pytorch.org/get-started/locally/) подходящие вам варианты операционной системы/видеокарты и скопируйте команду для установки. Разберем подробно самые популярные варианты установки:

### Установка в Linux ([поддерживаемые дистрибутивы](https://pytorch.org/get-started/locally/#supported-linux-distributions))

На линуксе будет работать поддержка `PyTorch` в любой конфигурации, что у вас нет видеокарты, что есть от Nvidia, что от AMD, или MPS apple M+.

Пререквизит для работы с видеокартой от Nvidia - нужно поставить CUDA, это инструмент от компании Nvidia, который позволяет ускорять вычисления на их же ГПУ. Чтобы поставить себе на машину все правильно воспользуйтесь этим [гайдом](https://docs.nvidia.com/cuda/cuda-installation-guide-linux/index.html) от Nvidia.

 - **pip**

`pip3 install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cpu` для тех, у кого нет видеокарты.

`pip3 install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu116` для тех, у кого есть видеокарта (либо другой `--extra-index-url`, смотрите на сайте PyTorch, в зависимости от версии CUDA).

 - **conda** / **mamba** https://github.com/conda-forge/miniforge#mambaforge

`conda install pytorch torchvision torchaudio cpuonly -c pytorch` для тех, у кого нет видеокарты.

`conda install pytorch torchvision torchaudio cudatoolkit=11.6 -c pytorch -c conda-forge` для тех, у кого есть видеокарта (либо немного другая команда, в зависимости от версии CUDA).

### Установка в Windows

На винде будет работать поддержка `PyTorch` только для видеокарт от Nvidia и без видеокарт вообще.

Пререквизит для работы с видеокартой от Nvidia - нужно поставить CUDA, это инструмент от компании Nvidia, который позволяет ускорять вычисления на их же ГПУ. Чтобы поставить себе на машину все правильно воспользуйтесь этим [гайдом](https://docs.nvidia.com/cuda/cuda-installation-guide-microsoft-windows/index.html) от Nvidia.

 - **pip**

`pip3 install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cpu` для тех, у кого нет видеокарты.

`pip3 install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu116` для тех, у кого есть видеокарта (либо другой `--extra-index-url`, смотрите на сайте PyTorch, в зависимости от версии CUDA).

 - **conda**

`conda install pytorch torchvision torchaudio cpuonly -c pytorch` для тех, у кого нет видеокарты.

`conda install pytorch torchvision torchaudio cudatoolkit=11.6 -c pytorch -c conda-forge` для тех, у кого есть видеокарта (либо немного другая команда, в зависимости от версии CUDA).



### Установка на Mac

На маках есть пока что поддержка `PyTorch` только центрального процессора, есть поддержка ускорения на чипах M1, M2, M1 Pro - "mps" https://docs.pytorch.org/docs/stable/notes/mps.html

 - **pip**

`pip3 install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cpu`

 - **conda**

`conda install pytorch torchvision torchaudio cpuonly -c pytorch`

In [1]:
!nvidia-smi

Fri Oct 24 11:45:45 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!nvtop

/bin/bash: line 1: nvtop: command not found


## Введение в `PyTorch`

### Тензоры

Тензоры — это специализированная структура данных, по сути это массивы и матрицы. Тензоры очень похожи на массивы в numpy, так что, если у вас хорошо с numpy, то разобраться в PyTorch тензорах будет очень просто. В PyTorch мы используем тензоры для кодирования входных и выходных данных модели, а также параметров модели.

In [3]:
import torch
import numpy as np

### Создание тензоров

Тензор можно создать напрямую из каких-то данных - нам подходят все списки с числами:

In [4]:
some_data = [1, 2, 3, 4]
some_tensor = torch.tensor(some_data)

some_tensor

tensor([1, 2, 3, 4])

In [5]:
some_data = [[1, 2], [3, 4], [5, 6]]
some_tensor = torch.tensor(some_data)

some_tensor

tensor([[1, 2],
        [3, 4],
        [5, 6]])

In [6]:
some_data = [[[1], [2]], [[3], [4]], [[5], [6]]]
some_tensor = torch.tensor(some_data)

some_tensor

tensor([[[1],
         [2]],

        [[3],
         [4]],

        [[5],
         [6]]])

На самом деле про "все" списки с числами - обман. Если у вашего списка есть какой-то уровень вложенности, то должны совпадать размерности у всех вложенных списков (подробнее про размерности поговорим позже):

In [7]:
some_other_data = [[1, 2], [3, 4], [5, 6, 7]]
some_other_tensor = torch.tensor(some_other_data)

some_other_tensor

ValueError: expected sequence of length 2 at dim 1 (got 3)

Также тензоры можно создавать из numpy массивов и наоборот:

In [8]:
some_numpy_array = np.array(some_data)

some_numpy_array

array([[[1],
        [2]],

       [[3],
        [4]],

       [[5],
        [6]]])

In [9]:
some_tensor_from_numpy = torch.from_numpy(some_numpy_array)

some_tensor_from_numpy

tensor([[[1],
         [2]],

        [[3],
         [4]],

        [[5],
         [6]]])

При этом если мы создаем тензор из numpy массива с помощью `torch.from_numpy`, то они делят между собой память, где лежат их данные и, соответственно, при изменении тензора меняется numpy массив и наоборот:

In [10]:
x = np.ones(10)
y = torch.from_numpy(x)

x, y

(array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]),
 tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.], dtype=torch.float64))

In [11]:
x += 1

x, y

(array([2., 2., 2., 2., 2., 2., 2., 2., 2., 2.]),
 tensor([2., 2., 2., 2., 2., 2., 2., 2., 2., 2.], dtype=torch.float64))

Можем создать тензор со случайными или константными значениями:

In [12]:
shape = (2, 3)

random_tensor = torch.rand(shape)
ones_tensor = torch.ones(shape)
zeros_tensor = torch.zeros(shape)
empty_tensor = torch.empty(shape) # uninitialized data is faster

random_tensor, ones_tensor, zeros_tensor, empty_tensor

(tensor([[0.4497, 0.2308, 0.5382],
         [0.7578, 0.0816, 0.5854]]),
 tensor([[1., 1., 1.],
         [1., 1., 1.]]),
 tensor([[0., 0., 0.],
         [0., 0., 0.]]),
 tensor([[9.1145e-18, 0.0000e+00, 9.1477e-41],
         [0.0000e+00, 1.4013e-44, 0.0000e+00]]))

In [13]:
torch.diag(torch.tensor([1, 2, 3]))

tensor([[1, 0, 0],
        [0, 2, 0],
        [0, 0, 3]])

Теперь поговорим про размерности подробнее.

У тензора есть какой-то размер, какая форма. Первое с чем нужно определиться, какой **размерности** тензор - количество осей у него.

In [14]:
shape = (10)  # одна ось (вектор)

tensor = torch.rand(shape)

tensor

tensor([0.2681, 0.5538, 0.7672, 0.7298, 0.0335, 0.2993, 0.2904, 0.3390, 0.9250,
        0.8378])

In [15]:
shape = (2, 3)  # две оси (матрица)

tensor = torch.rand(shape)

tensor

tensor([[0.5777, 0.7518, 0.9437],
        [0.0072, 0.5625, 0.6861]])

In [16]:
shape = (3, 2, 3)  # три оси (и больше - тензор)

tensor = torch.rand(shape)

tensor

tensor([[[0.3740, 0.3371, 0.5251],
         [0.8758, 0.0241, 0.3010]],

        [[0.0259, 0.0211, 0.8100],
         [0.5424, 0.8272, 0.6667]],

        [[0.0404, 0.1080, 0.9059],
         [0.6095, 0.7056, 0.2412]]])

Тензор с размерностью 1 - это просто вектор, список чисел.

Тензор с размерностью 2 - это просто матрица, то есть список списков чисел.

Тензор с размерностью 3 и больше - это тензор, то есть список списков списков ... чисел.

Получить доступ к размеру уже созданного тензора - метод `.shape`:

In [17]:
some_data = [[[1], [2]], [[3], [4]], [[5], [6]]]
some_tensor = torch.tensor(some_data)

print(some_tensor)
print(some_tensor.shape)

tensor([[[1],
         [2]],

        [[3],
         [4]],

        [[5],
         [6]]])
torch.Size([3, 2, 1])


In [18]:
torch.ones((3, 2, 1)).shape

torch.Size([3, 2, 1])

На первом семинаре по мо мы говорили про изображения, давайте сделаем тензор, который будет нам имитировать изображение - сделаем его размер `(c, h, w)`, где `h` и `w` это его высота и ширина, а `c` - число каналов в цветовом пространстве (в черно-белом 1, в RGB 3):

In [19]:
h = 9
w = 16
c = 3

shape = (c, h, w)

image_tensor = torch.rand(shape)

image_tensor

tensor([[[0.1611, 0.2846, 0.6813, 0.3854, 0.2014, 0.7443, 0.9021, 0.8548,
          0.7391, 0.1922, 0.8213, 0.5820, 0.7991, 0.4841, 0.7086, 0.6633],
         [0.0043, 0.1177, 0.8349, 0.6885, 0.6729, 0.0208, 0.1216, 0.1710,
          0.7042, 0.4011, 0.9372, 0.0588, 0.9499, 0.1472, 0.9047, 0.9479],
         [0.1396, 0.0829, 0.3355, 0.8304, 0.1338, 0.8493, 0.9149, 0.3728,
          0.5424, 0.1273, 0.2108, 0.7125, 0.2578, 0.8575, 0.0436, 0.3814],
         [0.8674, 0.0629, 0.9671, 0.8721, 0.3025, 0.3334, 0.8975, 0.4732,
          0.0481, 0.1026, 0.6313, 0.9392, 0.9357, 0.6375, 0.0189, 0.5283],
         [0.2422, 0.8491, 0.6334, 0.3199, 0.3688, 0.6173, 0.4411, 0.7923,
          0.5160, 0.2021, 0.8267, 0.8882, 0.8392, 0.5619, 0.6910, 0.8015],
         [0.1960, 0.3403, 0.7619, 0.2939, 0.7115, 0.8764, 0.3215, 0.2362,
          0.7339, 0.2654, 0.1524, 0.1561, 0.4193, 0.7901, 0.3987, 0.7147],
         [0.5144, 0.3183, 0.1672, 0.4785, 0.8636, 0.9884, 0.6011, 0.8234,
          0.2794, 0.6321, 0.9944

In [20]:
image_tensor.shape

torch.Size([3, 9, 16])

Можем попробовать поменять размер тензора, например, [вытянуть его в вектор](https://pytorch.org/docs/stable/generated/torch.ravel.html):

In [21]:
image_tensor.ravel() # the same memory

tensor([0.1611, 0.2846, 0.6813, 0.3854, 0.2014, 0.7443, 0.9021, 0.8548, 0.7391,
        0.1922, 0.8213, 0.5820, 0.7991, 0.4841, 0.7086, 0.6633, 0.0043, 0.1177,
        0.8349, 0.6885, 0.6729, 0.0208, 0.1216, 0.1710, 0.7042, 0.4011, 0.9372,
        0.0588, 0.9499, 0.1472, 0.9047, 0.9479, 0.1396, 0.0829, 0.3355, 0.8304,
        0.1338, 0.8493, 0.9149, 0.3728, 0.5424, 0.1273, 0.2108, 0.7125, 0.2578,
        0.8575, 0.0436, 0.3814, 0.8674, 0.0629, 0.9671, 0.8721, 0.3025, 0.3334,
        0.8975, 0.4732, 0.0481, 0.1026, 0.6313, 0.9392, 0.9357, 0.6375, 0.0189,
        0.5283, 0.2422, 0.8491, 0.6334, 0.3199, 0.3688, 0.6173, 0.4411, 0.7923,
        0.5160, 0.2021, 0.8267, 0.8882, 0.8392, 0.5619, 0.6910, 0.8015, 0.1960,
        0.3403, 0.7619, 0.2939, 0.7115, 0.8764, 0.3215, 0.2362, 0.7339, 0.2654,
        0.1524, 0.1561, 0.4193, 0.7901, 0.3987, 0.7147, 0.5144, 0.3183, 0.1672,
        0.4785, 0.8636, 0.9884, 0.6011, 0.8234, 0.2794, 0.6321, 0.9944, 0.3343,
        0.9501, 0.0563, 0.2717, 0.7051, 

In [22]:
image_tensor.ravel().shape

torch.Size([432])

In [23]:
h * w * c

432

Посчитаем количество элементов в тензоре с помощью [специальной функции](https://pytorch.org/docs/stable/generated/torch.numel.html):

In [24]:
image_tensor.numel()

432

In [25]:
h = 2
w = 3
c = 3

shape = (c, h, w)

image_tensor = torch.rand(shape)

image_tensor

tensor([[[0.7139, 0.4177, 0.4153],
         [0.8588, 0.1525, 0.8998]],

        [[0.0291, 0.3266, 0.3552],
         [0.5617, 0.2986, 0.3143]],

        [[0.2114, 0.8953, 0.2370],
         [0.7234, 0.6602, 0.5728]]])

Попробуем поменять размер с помощью функции [reshape](https://pytorch.org/docs/stable/generated/torch.reshape.html#torch.reshape):

In [26]:
image_tensor.reshape(c, h * w).shape # if not contiguous - create new memory

torch.Size([3, 6])

In [27]:
image_tensor.is_contiguous()

True

In [28]:
image_tensor.T.T.is_contiguous()

/tmp/ipython-input-2029451502.py:1: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4421.)
  image_tensor.T.is_contiguous()


False

Попробуем собрать из нескольких тензоров один большой:

[torch.cat](https://pytorch.org/docs/stable/generated/torch.cat.html#torch.cat)

In [29]:
x = torch.randn(2, 3)

In [30]:
x

tensor([[ 0.4914, -0.2501,  1.2879],
        [ 0.1164, -0.0407,  0.6373]])

In [31]:
torch.cat((x, x, x), dim=0).shape

torch.Size([6, 3])

In [32]:
torch.cat((x, x, x), dim=1).shape

torch.Size([2, 9])

In [33]:
x = torch.randn(3, 3)
y = torch.randn(5, 3)
z = torch.randn(1, 3)

for tensor in [x, y, z]:
    print(tensor)

torch.cat((x, y, z), dim=0).shape

tensor([[ 0.0933,  0.9905, -2.0270],
        [-0.1428,  0.5518, -1.1106],
        [ 0.3119, -0.2307,  0.4334]])
tensor([[-1.3157, -1.5267, -0.6681],
        [ 0.0104, -1.2590,  0.3221],
        [ 1.1385, -0.6940,  0.5996],
        [-0.0581,  0.0033, -1.1413],
        [-1.3510, -0.9821, -0.6639]])
tensor([[ 0.5458, -0.2799,  0.0085]])


torch.Size([9, 3])

In [34]:
x = torch.randn(2, 3)
y = torch.randn(2, 5)
z = torch.randn(2, 1)

for tensor in [x, y, z]:
    print(tensor)

torch.cat((x, y, z), dim=1).shape

tensor([[-3.3903,  0.0343,  0.4921],
        [-1.3179,  1.0123,  1.4409]])
tensor([[ 1.7705,  0.8806,  2.4811,  1.4100,  0.5499],
        [-0.9464, -1.8059,  0.0043, -0.3002,  0.0369]])
tensor([[1.5094],
        [0.1538]])


torch.Size([2, 9])

Теперь добавим дополнительную ось:

[torch.unsqueeze](https://pytorch.org/docs/stable/generated/torch.unsqueeze.html)

In [35]:
x = torch.rand(2, 3)

print(x)
print()
print(x.unsqueeze(0), x.unsqueeze(0).shape)
print()
print(x.unsqueeze(1), x.unsqueeze(1).shape)
print()
print(x.unsqueeze(2), x.unsqueeze(2).shape)
print()
print(x.unsqueeze(2), x.unsqueeze(-1).shape)

tensor([[0.4216, 0.3981, 0.1911],
        [0.8672, 0.4195, 0.7479]])

tensor([[[0.4216, 0.3981, 0.1911],
         [0.8672, 0.4195, 0.7479]]]) torch.Size([1, 2, 3])

tensor([[[0.4216, 0.3981, 0.1911]],

        [[0.8672, 0.4195, 0.7479]]]) torch.Size([2, 1, 3])

tensor([[[0.4216],
         [0.3981],
         [0.1911]],

        [[0.8672],
         [0.4195],
         [0.7479]]]) torch.Size([2, 3, 1])

tensor([[[0.4216],
         [0.3981],
         [0.1911]],

        [[0.8672],
         [0.4195],
         [0.7479]]]) torch.Size([2, 3, 1])


Уберем лишние оси (где размер единичка):

In [36]:
x = torch.rand(1, 2, 1, 3)

print(x)
print()
print(x.squeeze(), x.squeeze().shape)
print()
print(x.squeeze(0), x.squeeze(0).shape)

tensor([[[[0.9070, 0.1659, 0.4002]],

         [[0.8260, 0.1851, 0.0586]]]])

tensor([[0.9070, 0.1659, 0.4002],
        [0.8260, 0.1851, 0.0586]]) torch.Size([2, 3])

tensor([[[0.9070, 0.1659, 0.4002]],

        [[0.8260, 0.1851, 0.0586]]]) torch.Size([2, 1, 3])


Теперь поговорим про типы данных в тензорах. По умолчанию в тензорах лежат числа в torch.float32 для вещественных и torch.int64 для целочисленных.

In [37]:
tensor = torch.tensor([1.5, 2.2, 3.7, 4.9])

tensor

tensor([1.5000, 2.2000, 3.7000, 4.9000])

In [38]:
tensor.dtype

torch.float32

In [39]:
tensor = torch.tensor([1.5, 2.2, 3.7, 4.9], dtype=torch.float16)

tensor

tensor([1.5000, 2.1992, 3.6992, 4.8984], dtype=torch.float16)

In [40]:
tensor = torch.tensor([1.5, 2.2, 3.7, 4.9], dtype=torch.bfloat16)

tensor

tensor([1.5000, 2.2031, 3.7031, 4.9062], dtype=torch.bfloat16)

In [41]:
tensor = torch.tensor([1.5, 2.2, 3.7, 4.9], dtype=torch.float64)

tensor

tensor([1.5000, 2.2000, 3.7000, 4.9000], dtype=torch.float64)

In [42]:
tensor = torch.tensor([15, 22, 37, 49])

tensor

tensor([15, 22, 37, 49])

In [43]:
tensor.dtype

torch.int64

In [44]:
tensor = torch.tensor([15, 22, 37, 49], dtype=torch.int32)

tensor

tensor([15, 22, 37, 49], dtype=torch.int32)

In [45]:
tensor = torch.tensor([15, 22, 37, 49], dtype=torch.int16)

tensor

tensor([15, 22, 37, 49], dtype=torch.int16)

Размещение тензора на GPU:

In [46]:
!nvidia-smi

Fri Oct 24 11:59:11 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [47]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name())

True
Tesla T4


In [48]:
!nvidia-smi

Fri Oct 24 11:59:28 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [50]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

print(device)

cuda:0


In [51]:
tensor = torch.tensor([15, 22, 37, 49], device=device)

tensor

tensor([15, 22, 37, 49], device='cuda:0')

In [52]:
tensor = torch.tensor([15, 22, 37, 49])

print(tensor)

tensor = tensor.to(device)

tensor

tensor([15, 22, 37, 49])


tensor([15, 22, 37, 49], device='cuda:0')

In [53]:
tensor.to(torch.int32)

tensor([15, 22, 37, 49], device='cuda:0', dtype=torch.int32)

In [54]:
tensor = tensor.cpu()

tensor

tensor([15, 22, 37, 49])

In [55]:
tensor.cuda()

tensor([15, 22, 37, 49], device='cuda:0')

In [56]:
tensor.device

device(type='cpu')

In [57]:
a = torch.rand(2, 3)
b = torch.rand(2, 3)

a + b

tensor([[1.0949, 1.8928, 1.1455],
        [1.1961, 0.5459, 0.7085]])

In [58]:
a = a.to(device)

a + b

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

In [59]:
b = b.to(device)

a + b

tensor([[1.0949, 1.8928, 1.1455],
        [1.1961, 0.5459, 0.7085]], device='cuda:0')

### Операции с тензорами

Большая часть операций с тензорами хорошо описана в их [документации](https://pytorch.org/docs/stable/torch.html), разберем основные:

In [60]:
a = torch.rand(2, 3)
b = torch.rand(2, 3)

a, b

(tensor([[0.4832, 0.4642, 0.9696],
         [0.0428, 0.1744, 0.7082]]),
 tensor([[0.1932, 0.5861, 0.3618],
         [0.6330, 0.4342, 0.7337]]))

In [61]:
# поэлементные

print(a + b)

print()

print(torch.add(a, b))

print()

print(a.add(b))

tensor([[0.6763, 1.0503, 1.3314],
        [0.6759, 0.6086, 1.4419]])

tensor([[0.6763, 1.0503, 1.3314],
        [0.6759, 0.6086, 1.4419]])

tensor([[0.6763, 1.0503, 1.3314],
        [0.6759, 0.6086, 1.4419]])


In [62]:
print(a - b)

print()

print(torch.sub(a, b))

print()

print(a.sub(b))

tensor([[ 0.2900, -0.1220,  0.6078],
        [-0.5902, -0.2598, -0.0255]])

tensor([[ 0.2900, -0.1220,  0.6078],
        [-0.5902, -0.2598, -0.0255]])

tensor([[ 0.2900, -0.1220,  0.6078],
        [-0.5902, -0.2598, -0.0255]])


In [63]:
print(a * b)

print()

print(torch.mul(a, b))

print()

print(a.mul(b))

print()

# print(a.mul_(b))

tensor([[0.0933, 0.2721, 0.3508],
        [0.0271, 0.0757, 0.5196]])

tensor([[0.0933, 0.2721, 0.3508],
        [0.0271, 0.0757, 0.5196]])

tensor([[0.0933, 0.2721, 0.3508],
        [0.0271, 0.0757, 0.5196]])



In [64]:
print(a / b)

print()

print(torch.div(a, b))

print()

print(a.div(b))

tensor([[2.5015, 0.7919, 2.6801],
        [0.0677, 0.4016, 0.9653]])

tensor([[2.5015, 0.7919, 2.6801],
        [0.0677, 0.4016, 0.9653]])

tensor([[2.5015, 0.7919, 2.6801],
        [0.0677, 0.4016, 0.9653]])


In [65]:
a = torch.rand(2, 3)
b = torch.rand(3, 4)
c = torch.rand(5, 5)

a, b, c

(tensor([[0.3853, 0.9055, 0.9601],
         [0.6574, 0.0013, 0.0739]]),
 tensor([[0.9407, 0.1879, 0.9008, 0.7747],
         [0.6511, 0.3105, 0.3483, 0.1535],
         [0.9727, 0.2201, 0.9513, 0.9220]]),
 tensor([[0.0149, 0.4187, 0.9743, 0.7457, 0.5777],
         [0.7245, 0.8337, 0.6751, 0.8972, 0.9521],
         [0.6741, 0.0626, 0.8180, 0.6971, 0.2324],
         [0.4037, 0.5518, 0.4158, 0.3215, 0.3035],
         [0.8736, 0.1931, 0.8859, 0.9493, 0.5456]]))

In [66]:
# матричные операции

print(a @ b, (a @ b).shape)

print()

print(torch.matmul(a, b), torch.matmul(a, b).shape)

print()

print(c.trace())

print()

print(c.exp())

tensor([[1.8858, 0.5649, 1.5758, 1.3227],
        [0.6911, 0.1402, 0.6630, 0.5776]]) torch.Size([2, 4])

tensor([[1.8858, 0.5649, 1.5758, 1.3227],
        [0.6911, 0.1402, 0.6630, 0.5776]]) torch.Size([2, 4])

tensor(2.5337)

tensor([[1.0150, 1.5199, 2.6493, 2.1080, 1.7819],
        [2.0637, 2.3018, 1.9643, 2.4527, 2.5913],
        [1.9622, 1.0647, 2.2660, 2.0080, 1.2617],
        [1.4973, 1.7364, 1.5155, 1.3792, 1.3546],
        [2.3954, 1.2130, 2.4251, 2.5839, 1.7257]])


### [Автоматическое дифференцирование](https://pytorch.org/docs/stable/notes/autograd.html)

про leaf nodes https://docs.pytorch.org/tutorials/beginner/understanding_leaf_vs_nonleaf_tutorial.html

In [17]:
x = torch.rand(5)

x

tensor([0.8916, 0.4843, 0.8047, 0.1836, 0.2450])

In [16]:
w = torch.rand(3, 5, requires_grad=True)

w

tensor([[0.0199, 0.4449, 0.0162, 0.5221, 0.7964],
        [0.9870, 0.2734, 0.5040, 0.9278, 0.7685],
        [0.6915, 0.2650, 0.7752, 0.9648, 0.3679]], requires_grad=True)

In [18]:
print(w.grad)

None


In [19]:
z = torch.matmul(x, w.t())

z

tensor([0.5372, 1.7767, 1.6360], grad_fn=<SqueezeBackward4>)

In [20]:
v = torch.rand(3, requires_grad=True)

v

tensor([0.1872, 0.0938, 0.2575], requires_grad=True)

In [21]:
print(v.grad)

None


In [22]:
y = torch.sum(z * v)

y

tensor(0.6885, grad_fn=<SumBackward0>)

In [23]:
y.item() # extract scalar from gpu

0.6884597539901733

In [24]:
loss = torch.mean((y - 2) ** 2)

In [25]:
loss

tensor(1.7201, grad_fn=<MeanBackward0>)

In [77]:
print(f'{x.grad=}\n')
print(f'{w.grad=}\n')
print(f'{z.grad=}\n')
print(f'{v.grad=}\n')

x.grad=None

w.grad=None

z.grad=None

v.grad=None



/tmp/ipython-input-162867685.py:3: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:489.)
  print(f'{z.grad=}\n')


По дефолту Pytorch не дает вам доступ к градиентам промежуточных нод в графе, но можно его получить, если сделать tensor.retain_grad()

In [26]:
loss.backward()

In [27]:
print(f'{x.grad=}\n')
print(f'{w.grad=}\n')
print(f'{z.grad=}\n')
print(f'{v.grad=}\n')

x.grad=None

w.grad=tensor([[-0.4378, -0.2378, -0.3951, -0.0901, -0.1203],
        [-0.2193, -0.1191, -0.1979, -0.0451, -0.0602],
        [-0.6023, -0.3272, -0.5436, -0.1240, -0.1655]])

z.grad=None

v.grad=tensor([-1.4092, -4.6603, -4.2912])



/tmp/ipython-input-162867685.py:3: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:489.)
  print(f'{z.grad=}\n')


In [37]:
w.retain_grad=True

In [43]:
w.shape

torch.Size([3, 5])

In [46]:
w.permute(1, 0)

None


/tmp/ipython-input-292015073.py:1: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:489.)
  print(w.permute(1, 0).grad)


In [39]:
t = w.T
t

tensor([[0.0199, 0.9870, 0.6915],
        [0.4449, 0.2734, 0.2650],
        [0.0162, 0.5040, 0.7752],
        [0.5221, 0.9278, 0.9648],
        [0.7964, 0.7685, 0.3679]], grad_fn=<PermuteBackward0>)

In [47]:
print(w.grad)

tensor([[-0.4378, -0.2378, -0.3951, -0.0901, -0.1203],
        [-0.2193, -0.1191, -0.1979, -0.0451, -0.0602],
        [-0.6023, -0.3272, -0.5436, -0.1240, -0.1655]])


In [42]:
t.shape, y.shape

(torch.Size([5, 3]), torch.Size([]))

In [15]:
a = torch.rand(1, requires_grad=True)
b = torch.rand(1, requires_grad=True)

a, b

(tensor([0.7432], requires_grad=True), tensor([0.5362], requires_grad=True))

In [81]:
loss = (a - b)

loss

tensor([-0.5660], grad_fn=<SubBackward0>)

In [82]:
print(f'{a.grad=}\n')
print(f'{b.grad=}\n')

a.grad=None

b.grad=None



In [83]:
loss.backward()

In [84]:
print(f'{a.grad=}\n')  # 1
print(f'{b.grad=}\n')  # -1

a.grad=tensor([1.])

b.grad=tensor([-1.])



In [85]:
a.grad.zero_()
b.grad.zero_()

tensor([0.])

In [86]:
loss = (a - b) ** 2

loss

tensor([0.3203], grad_fn=<PowBackward0>)

In [87]:
print(f'{a.grad=}\n')
print(f'{b.grad=}\n')

a.grad=tensor([0.])

b.grad=tensor([0.])



In [88]:
loss.backward()

In [89]:
print(f'{a.grad=}\n')  # 2 * (a - b)
print(f'{b.grad=}\n')  # -2 * (a - b)

a.grad=tensor([-1.1319])

b.grad=tensor([1.1319])



In [90]:
2 * (a - b)

tensor([-1.1319], grad_fn=<MulBackward0>)

In [91]:
a = torch.rand(3, 5, requires_grad=True)
b = torch.rand(3, 5, requires_grad=True)

a, b

(tensor([[0.2776, 0.0480, 0.8636, 0.5177, 0.3422],
         [0.5157, 0.0309, 0.2680, 0.7097, 0.5813],
         [0.5721, 0.6808, 0.5849, 0.2924, 0.9258]], requires_grad=True),
 tensor([[0.3498, 0.0337, 0.2666, 0.4585, 0.0073],
         [0.3544, 0.3299, 0.6703, 0.2958, 0.3452],
         [0.2825, 0.5669, 0.7669, 0.6531, 0.0389]], requires_grad=True))

In [92]:
loss = torch.mean(a * b)

loss

tensor(0.1717, grad_fn=<MeanBackward0>)

In [93]:
print(f'{a.grad=}\n')
print(f'{b.grad=}\n')

a.grad=None

b.grad=None



In [94]:
loss.backward()

In [95]:
print(f'{a.grad=}\n')  # b / (3 * 5)
print(f'{b.grad=}\n')  # a / (3 * 5)

a.grad=tensor([[0.0233, 0.0022, 0.0178, 0.0306, 0.0005],
        [0.0236, 0.0220, 0.0447, 0.0197, 0.0230],
        [0.0188, 0.0378, 0.0511, 0.0435, 0.0026]])

b.grad=tensor([[0.0185, 0.0032, 0.0576, 0.0345, 0.0228],
        [0.0344, 0.0021, 0.0179, 0.0473, 0.0388],
        [0.0381, 0.0454, 0.0390, 0.0195, 0.0617]])



In [96]:
a / 15

tensor([[0.0185, 0.0032, 0.0576, 0.0345, 0.0228],
        [0.0344, 0.0021, 0.0179, 0.0473, 0.0388],
        [0.0381, 0.0454, 0.0390, 0.0195, 0.0617]], grad_fn=<DivBackward0>)

In [97]:
b / 15

tensor([[0.0233, 0.0022, 0.0178, 0.0306, 0.0005],
        [0.0236, 0.0220, 0.0447, 0.0197, 0.0230],
        [0.0188, 0.0378, 0.0511, 0.0435, 0.0026]], grad_fn=<DivBackward0>)

In [98]:
a = torch.rand(3, 5, requires_grad=True)

print(f'{a=}\n')

loss1 = torch.sum(a ** 2) # 2a
loss2 = torch.sum(a) # 1

print(f'{a.grad=}\n')

loss1.backward()

print(f'{a.grad=}\n')

loss2.backward()

print(f'{a.grad=}\n')

a=tensor([[0.7186, 0.9155, 0.8969, 0.5277, 0.6048],
        [0.1168, 0.2634, 0.4411, 0.3151, 0.8008],
        [0.5885, 0.8188, 0.1086, 0.7817, 0.2454]], requires_grad=True)

a.grad=None

a.grad=tensor([[1.4373, 1.8310, 1.7938, 1.0554, 1.2097],
        [0.2336, 0.5268, 0.8822, 0.6301, 1.6017],
        [1.1769, 1.6376, 0.2172, 1.5633, 0.4908]])

a.grad=tensor([[2.4373, 2.8310, 2.7938, 2.0554, 2.2097],
        [1.2336, 1.5268, 1.8822, 1.6301, 2.6017],
        [2.1769, 2.6376, 1.2172, 2.5633, 1.4908]])



In [99]:
print(f'{2*a=}\n')
print(f'{2*a+1=}')

2*a=tensor([[1.4373, 1.8310, 1.7938, 1.0554, 1.2097],
        [0.2336, 0.5268, 0.8822, 0.6301, 1.6017],
        [1.1769, 1.6376, 0.2172, 1.5633, 0.4908]], grad_fn=<MulBackward0>)

2*a+1=tensor([[2.4373, 2.8310, 2.7938, 2.0554, 2.2097],
        [1.2336, 1.5268, 1.8822, 1.6301, 2.6017],
        [2.1769, 2.6376, 1.2172, 2.5633, 1.4908]], grad_fn=<AddBackward0>)


In [100]:
a = torch.rand(3, 5, requires_grad=True)
b = torch.rand(3, 5, requires_grad=False)

a, b

(tensor([[0.4259, 0.3011, 0.3757, 0.5438, 0.0119],
         [0.6586, 0.9271, 0.8768, 0.0623, 0.7579],
         [0.3417, 0.7943, 0.9374, 0.9839, 0.0438]], requires_grad=True),
 tensor([[0.6809, 0.8208, 0.4898, 0.8150, 0.6145],
         [0.5289, 0.1088, 0.8319, 0.3103, 0.8779],
         [0.4550, 0.9173, 0.7136, 0.9866, 0.3939]]))

In [101]:
loss = torch.sum(a - b)

loss

tensor(-1.5026, grad_fn=<SumBackward0>)

In [102]:
print(f'{a.grad=}\n')
print(f'{b.grad=}\n')

a.grad=None

b.grad=None



In [103]:
loss.backward()

In [104]:
print(f'{a.grad=}\n')  # all ones
print(f'{b.grad=}\n')  # None

a.grad=tensor([[1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.]])

b.grad=None



In [105]:
a = torch.rand(3, 5, requires_grad=True)
b = torch.rand(3, 5, requires_grad=True)

a, b

(tensor([[0.2888, 0.7448, 0.6162, 0.3207, 0.5074],
         [0.0733, 0.0257, 0.4952, 0.6573, 0.7214],
         [0.2089, 0.2738, 0.3945, 0.8736, 0.3930]], requires_grad=True),
 tensor([[0.1440, 0.9033, 0.6889, 0.2091, 0.2558],
         [0.9439, 0.5819, 0.4350, 0.1634, 0.4406],
         [0.9251, 0.0100, 0.0436, 0.3629, 0.4247]], requires_grad=True))

https://docs.pytorch.org/docs/stable/notes/autograd.html#locally-disable-grad-doc

In [106]:
loss = torch.sum(a - b)
loss

tensor(0.0624, grad_fn=<SumBackward0>)

In [107]:
loss

tensor(0.0624, grad_fn=<SumBackward0>)

In [108]:
loss.backward()

In [109]:
a.grad, b.grad

(tensor([[1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.]]),
 tensor([[-1., -1., -1., -1., -1.],
         [-1., -1., -1., -1., -1.],
         [-1., -1., -1., -1., -1.]]))

In [110]:
a = torch.rand(3, 5, requires_grad=True)
b = torch.rand(3, 5, requires_grad=True)

a, b

(tensor([[0.6294, 0.6155, 0.0039, 0.4150, 0.9943],
         [0.9715, 0.2847, 0.5451, 0.8756, 0.3077],
         [0.7023, 0.0893, 0.3081, 0.3137, 0.7102]], requires_grad=True),
 tensor([[0.5766, 0.6323, 0.4663, 0.6693, 0.2829],
         [0.0062, 0.8462, 0.4095, 0.4381, 0.0017],
         [0.6589, 0.7670, 0.6238, 0.1681, 0.3706]], requires_grad=True))

In [111]:
with torch.inference_mode():
    loss = torch.sum(a - b)

loss

tensor(0.8488)

In [112]:
loss.backward()

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [113]:
with torch.no_grad():
    a = torch.rand(3, 5, requires_grad=True)
    b = torch.rand(3, 5, requires_grad=True)

    loss = torch.sum(a + b)

    print(f'{loss=}')

    loss.backward()

loss=tensor(15.6894)


RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [114]:
@torch.no_grad()
def foo():
    a = torch.rand(3, 5, requires_grad=True)
    b = torch.rand(3, 5, requires_grad=True)

    loss = torch.mean(a + b)

    print(f'{loss=}')

    return a, b

In [115]:
a, b = foo()

loss=tensor(1.0497)


In [116]:
torch.mean(a - b)

tensor(-0.0447, grad_fn=<MeanBackward0>)

In [117]:
@torch.inference_mode()
def foo():
    a = torch.rand(3, 5, requires_grad=True)
    b = torch.rand(3, 5, requires_grad=True)

    loss = torch.mean(a + b)

    print(f'{loss=}')

    return a, b

In [118]:
a, b = foo()

loss=tensor(0.9571)


In [119]:
torch.mean(a - b)

tensor(0.1626)

## Полносвязные слои и функции активации в `PyTorch`

In [120]:
from torch import nn

In [121]:
import torch.nn as nn

### Полносвязный слой

>$y_j = \sum\limits_{i=1}^{n}x_iw_{ji} + b_j$


In [122]:
layer = nn.Linear(in_features=5, out_features=3)

In [123]:
layer

Linear(in_features=5, out_features=3, bias=True)

In [124]:
layer.weight

Parameter containing:
tensor([[-0.1613, -0.2048,  0.3340,  0.3528, -0.2596],
        [ 0.2980,  0.3407, -0.0085,  0.3101, -0.4264],
        [ 0.3497, -0.2711,  0.2620, -0.1182,  0.3997]], requires_grad=True)

In [125]:
layer.weight.shape

torch.Size([3, 5])

In [126]:
layer.bias

Parameter containing:
tensor([ 0.3081, -0.1886,  0.0124], requires_grad=True)

In [127]:
layer.bias.shape

torch.Size([3])

In [128]:
layer = nn.Linear(in_features=5, out_features=3, bias=False)

In [129]:
layer.bias is None

True

In [130]:
# layer.__call__

In [131]:
x = torch.randn(5)

print(layer(x))

tensor([0.3130, 0.6530, 0.4919], grad_fn=<SqueezeBackward4>)


In [132]:
layer1 = nn.Linear(in_features=5, out_features=3)
layer2 = nn.Linear(in_features=3, out_features=1)

layer2(layer1(x))

tensor([-0.0129], grad_fn=<ViewBackward0>)

### Функции активации

> Сигмоида $f(x) = \dfrac{1}{1 + e^{-x}}$

In [133]:
activation = nn.Sigmoid()

In [134]:
x = torch.randn(5)

print(x)

print(activation(x))

tensor([-0.5285,  1.4690,  3.2051, -0.7655,  0.4065])
tensor([0.3709, 0.8129, 0.9610, 0.3174, 0.6003])


> ReLU $f(x) = \max(0, x)$

In [135]:
activation = nn.ReLU()

In [136]:
x = torch.randn(5)

print(x)

print(activation(x))

tensor([ 0.3333, -1.1622, -0.6535, -0.5167, -0.7036])
tensor([0.3333, 0.0000, 0.0000, 0.0000, 0.0000])


> Leaky ReLU $f(x) = \max(0, x) + \alpha \min(0, x)$

In [140]:
activation = nn.LeakyReLU(negative_slope=0.001)

In [141]:
x = torch.randn(5)

print(x)

print(activation(x))

tensor([ 0.4154, -1.3276,  0.2397,  0.3472, -0.5833])
tensor([ 0.4154, -0.0013,  0.2397,  0.3472, -0.0006])


In [142]:
layer1 = nn.Linear(in_features=5, out_features=3)
activation = nn.LeakyReLU(negative_slope=0.001)
layer2 = nn.Linear(in_features=3, out_features=1)

layer2(activation(layer1(x)))

tensor([-0.1315], grad_fn=<ViewBackward0>)

## Градиентный спуск своими руками

In [143]:
n_features = 2
n_objects = 300

torch.manual_seed(0)

w_true = torch.randn(n_features)
b_true = torch.randn(1)

x = (torch.rand(n_objects, n_features) - 0.5) * 10 * (torch.arange(n_features) * 2 + 1)
y = torch.matmul(x, w_true) + torch.randn(n_objects) + b_true

In [144]:
x.shape

torch.Size([300, 2])

In [145]:
y.shape

torch.Size([300])

In [146]:
n_steps = 200
step_size = 1e-2

In [148]:
w = torch.rand(n_features, requires_grad=True)
b = torch.rand(1, requires_grad=True)

for i in range(n_steps):
    y_pred = torch.matmul(x, w) + b

    mse = torch.mean((y_pred - y) ** 2)

    if i < 20 or i % 10 == 0:
        print(f'MSE на шаге {i + 1} {mse.item():.5f}')

    mse.backward()

#     print(f'{w.grad=}\n')
#     print(f'{b.grad=}\n')

    with torch.no_grad():
        w -= w.grad * step_size
        b -= b.grad * step_size

    # w.grad.zero_()
    # b.grad.zero_()

MSE на шаге 1 130.75197
MSE на шаге 2 53.79208
MSE на шаге 3 169.93105
MSE на шаге 4 7.26859
MSE на шаге 5 163.94931
MSE на шаге 6 46.82865
MSE на шаге 7 121.65747
MSE на шаге 8 127.41590
MSE на шаге 9 50.39906
MSE на шаге 10 168.74312
MSE на шаге 11 3.35809
MSE на шаге 12 156.36807
MSE на шаге 13 39.84022
MSE на шаге 14 113.91908
MSE на шаге 15 127.51176
MSE на шаге 16 53.44720
MSE на шаге 17 177.22952
MSE на шаге 18 11.58336
MSE на шаге 19 161.38419
MSE на шаге 20 43.86105
MSE на шаге 21 114.07033
MSE на шаге 31 182.86505
MSE на шаге 41 46.94762
MSE на шаге 51 36.81902
MSE на шаге 61 175.22643
MSE на шаге 71 124.89193
MSE на шаге 81 12.52258
MSE на шаге 91 111.42495
MSE на шаге 101 167.11227
MSE на шаге 111 68.61166
MSE на шаге 121 37.34223
MSE на шаге 131 150.03690
MSE на шаге 141 140.46170
MSE на шаге 151 33.03978
MSE на шаге 161 76.04433
MSE на шаге 171 174.18114
MSE на шаге 181 96.83352
MSE на шаге 191 11.35149


In [150]:
layer = nn.Linear(in_features=n_features, out_features=1)


for i in range(n_steps):
    y_pred = layer(x)

    mse = torch.mean((y_pred - y) ** 2) # [300, 1] - [300]

    if i < 20 or i % 10 == 0:
        print(f'MSE на шаге {i + 1} {mse.item():.5f}')

    mse.backward()

    with torch.no_grad():
        layer.weight -= layer.weight.grad * step_size
        layer.bias -= layer.bias.grad * step_size

    layer.weight.grad.zero_()
    layer.bias.grad.zero_()

    layer.zero_grad()

MSE на шаге 1 71.26740
MSE на шаге 2 46.77110
MSE на шаге 3 38.91867
MSE на шаге 4 36.19193
MSE на шаге 5 35.06435
MSE на шаге 6 34.45151
MSE на шаге 7 34.01669
MSE на шаге 8 33.65302
MSE на шаге 9 33.32498
MSE на шаге 10 33.01966
MSE на шаге 11 32.73158
MSE на шаге 12 32.45794
MSE на шаге 13 32.19704
MSE на шаге 14 31.94769
MSE на шаге 15 31.70901
MSE на шаге 16 31.48026
MSE на шаге 17 31.26088
MSE на шаге 18 31.05036
MSE на шаге 19 30.84825
MSE на шаге 20 30.65417
MSE на шаге 21 30.46776
MSE на шаге 31 28.96191
MSE на шаге 41 27.95099
MSE на шаге 51 27.27213
MSE на шаге 61 26.81626
MSE на шаге 71 26.51012
MSE на шаге 81 26.30454
MSE на шаге 91 26.16649
MSE на шаге 101 26.07378
MSE на шаге 111 26.01152
MSE на шаге 121 25.96971
MSE на шаге 131 25.94164
MSE на шаге 141 25.92278
MSE на шаге 151 25.91012
MSE на шаге 161 25.90162
MSE на шаге 171 25.89591
MSE на шаге 181 25.89208
MSE на шаге 191 25.88950


In [151]:
layer(x)

tensor([[-2.7104],
        [-2.7095],
        [-2.7134],
        [-2.7124],
        [-2.7074],
        [-2.7120],
        [-2.7094],
        [-2.7152],
        [-2.7101],
        [-2.6907],
        [-2.7127],
        [-2.7181],
        [-2.7114],
        [-2.7086],
        [-2.7202],
        [-2.7060],
        [-2.7032],
        [-2.6989],
        [-2.7119],
        [-2.7047],
        [-2.7122],
        [-2.7075],
        [-2.7040],
        [-2.6958],
        [-2.6999],
        [-2.7115],
        [-2.7053],
        [-2.7163],
        [-2.7005],
        [-2.7087],
        [-2.7075],
        [-2.7135],
        [-2.7091],
        [-2.7041],
        [-2.6990],
        [-2.7020],
        [-2.7243],
        [-2.6918],
        [-2.7042],
        [-2.7061],
        [-2.6990],
        [-2.7070],
        [-2.7022],
        [-2.7139],
        [-2.7034],
        [-2.7121],
        [-2.7018],
        [-2.7106],
        [-2.7180],
        [-2.7104],
        [-2.7092],
        [-2.7164],
        [-2.

In [152]:
layer(x).shape

torch.Size([300, 1])

In [153]:
y.shape

torch.Size([300])

In [154]:
(layer(x) - y).shape

torch.Size([300, 300])

In [155]:
layer(x).ravel().shape

torch.Size([300])

In [156]:
(layer(x).ravel() - y).shape

torch.Size([300])

In [157]:
layer = nn.Linear(in_features=n_features, out_features=1)

for i in range(n_steps):
    y_pred = layer(x).ravel()

    mse = torch.mean((y_pred - y) ** 2)

    if i < 20 or i % 10 == 0:
        print(f'MSE на шаге {i + 1} {mse.item():.5f}')

    mse.backward()

    with torch.no_grad():
        layer.weight -= layer.weight.grad * step_size
        layer.bias -= layer.bias.grad * step_size

    layer.zero_grad()

MSE на шаге 1 28.00511
MSE на шаге 2 18.32802
MSE на шаге 3 13.78828
MSE на шаге 4 11.26644
MSE на шаге 5 9.67071
MSE на шаге 6 8.57205
MSE на шаге 7 7.77375
MSE на шаге 8 7.17067
MSE на шаге 9 6.69975
MSE на шаге 10 6.32047
MSE на шаге 11 6.00582
MSE на шаге 12 5.73747
MSE на шаге 13 5.50286
MSE на шаге 14 5.29334
MSE на шаге 15 5.10291
MSE на шаге 16 4.92739
MSE на шаге 17 4.76383
MSE на шаге 18 4.61016
MSE на шаге 19 4.46489
MSE на шаге 20 4.32693
MSE на шаге 21 4.19549
MSE на шаге 31 3.14931
MSE на шаге 41 2.45062
MSE на шаге 51 1.98151
MSE на шаге 61 1.66649
MSE на шаге 71 1.45495
MSE на шаге 81 1.31288
MSE на шаге 91 1.21749
MSE на шаге 101 1.15342
MSE на шаге 111 1.11040
MSE на шаге 121 1.08151
MSE на шаге 131 1.06211
MSE на шаге 141 1.04908
MSE на шаге 151 1.04033
MSE на шаге 161 1.03446
MSE на шаге 171 1.03051
MSE на шаге 181 1.02786
MSE на шаге 191 1.02608


In [158]:
n_features = 5
n_objects = 300

torch.manual_seed(0)

w_true = torch.randn(n_features)
b_true = torch.randn(1)

x = (torch.rand(n_objects, n_features) - 0.5) * 10 * (torch.arange(n_features) * 2 + 1)
y = torch.matmul(x, w_true) + torch.randn(n_objects) + b_true

In [160]:
n_steps = 1000
step_size = 3e-4

w - w.grad * lr

In [163]:
layer1 = nn.Linear(in_features=n_features, out_features=3)
layer2 = nn.Linear(in_features=3, out_features=1)
activation = nn.ReLU()

for i in range(n_steps):
    y_pred = layer2(activation(layer1(x))).ravel()

    mse = torch.mean((y_pred - y) ** 2)

    if i < 20 or i % 50 == 0:
        print(f'MSE на шаге {i + 1} {mse.item():.5f}')

    mse.backward()

    with torch.no_grad():
        layer1.weight -= layer1.weight.grad * step_size
        layer1.bias -= layer1.bias.grad * step_size
        layer2.weight -= layer2.weight.grad * step_size
        layer2.bias -= layer2.bias.grad * step_size

    layer1.zero_grad()
    layer2.zero_grad()

MSE на шаге 1 2090.00342


RuntimeError: a leaf Variable that requires grad is being used in an in-place operation.

## Подробнее о view, reshape, contiguous и устройстве тензоров

In [164]:
x = torch.arange(2 * 3).reshape(2, 3)
x

tensor([[0, 1, 2],
        [3, 4, 5]])

Грубо говоря, внутри тензор устроен следующим образом. Сырые данные хранятся в одномерном массиве размера количества элементов в тензоре. Также необходимо хранить размер (shape), в данном случае (2, 3), и страйды (stride). Страйды это массив длины размерности тензора, где на i-ой позиции стоит величина, которую надо прибавить к индексу в одномерном массиве, чтобы увеличить индекс в многомерном массиве на 1.

Для матрицы (2,3) страйды будут выглядеть как (3,1). Чтобы перейти на следующую строку нам нужно прибавить 3 к индексу в массиве, чтобы перейти к следующему столбцу добавляем 1 к индексу.

Это все обобщается на многомерный случай аналогично.

In [165]:
x.storage()

/tmp/ipython-input-2674773268.py:1: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  x.storage()


 0
 1
 2
 3
 4
 5
[torch.storage.TypedStorage(dtype=torch.int64, device=cpu) of size 6]

In [166]:
x.stride()

(3, 1)

То есть в stride[i] лежит число - которое нужно прибавить к индексу внутри storage данных чтобы получить в dim i следующий элемент

В данном случае данные лежат "непрерывно" в одномерном массиве.

In [167]:
x.is_contiguous()

True

Зачем все-таки нужны страйды? Чтобы забесплатно производить решейпинг тензора, транспонирование и тд, меняя только этот массив strides, но не копируя сами сырые данные.

Чтобы транспонировать матрицу, достаточно поменять местами страйды у строк и столбцов.

In [168]:
y = x.permute(1, 0)
y

tensor([[0, 3],
        [1, 4],
        [2, 5]])

In [172]:
x.is_contiguous()

True

In [169]:
y.is_contiguous()

False

Данные не изменились.

In [170]:
y.storage()

 0
 1
 2
 3
 4
 5
[torch.storage.TypedStorage(dtype=torch.int64, device=cpu) of size 6]

In [171]:
y.stride()

(1, 3)

Делать view можно только на непрерывный тензор. Решейп в свою очередь "онепрерывит" тензор, скопирует сырые данные и упакует их правильным образом (.contiguous()), а потом уже поменяет шейпы.

In [173]:
y.view(2, 3)

RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.

In [174]:
y.reshape(2, 3)

tensor([[0, 3, 1],
        [4, 2, 5]])

In [176]:
y.contiguous().view(2, 3)

tensor([[0, 3, 1],
        [4, 2, 5]])

In [177]:
y.contiguous().stride()

(2, 1)

Данные скопирировались после .contiguous()

In [178]:
y.contiguous().storage()

 0
 3
 1
 4
 2
 5
[torch.storage.TypedStorage(dtype=torch.int64, device=cpu) of size 6]

Работа с непрерывными тензорами быстрее, так как необязательно постоянно сопоставлять мульти-индекс тензора с индексом в одномерном массиве. Если операции поэлементные, то можно просто работать с исходным массивом сырых данных.

Однако, надо иметь в виду, что .contiguous() копирует данные, что может быть ботлнеком в некоторых сценариях.

In [10]:
import torch

x = torch.rand(3, 4, 4)
x.shape

torch.Size([3, 4, 4])

In [13]:
x.permute(0, 2, 1).is_contiguous()

False

In [14]:
x.stride()

(16, 4, 1)